In [27]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn # To ignore some warnings from sklearn logistic regression


from datetime import datetime
import numpy as np
from scipy.optimize import minimize # minimizing function for AIOLI
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.linear_model import LogisticRegression # for classic logistic regression
from typing import Literal

rng = np.random.default_rng() # To generate random numbers
options={"disp": False} # To not get verbose from minimize function

## Functions for algorithms

First I define some functions that will be used later in all algorithms.

In [28]:
def sig(z):
    """ Sigmoid function that avoids overflow """
    if z < 0:
        return np.exp(z) / (1 + np.exp(z))
    else:
        return 1 / (1 + np.exp(-z))

In [29]:
# NOTE TO DO : put in general algorithm function that if radius == NA then this is not done.
def project_onto_l2_ball(x,radius):
    """ Euclidean projection onto L2 ball"""
    norm = np.linalg.norm(x)
    if norm <= radius:
        return x
    return x * radius/norm

In [30]:
def calculate_OTB(array, method: Literal["suffix_averaging", "uniform", "last_iterate"] | None = None, p = 0.5):
    """ Weighted average function
    ----
    Parameters:
        p: Proportion of last observations to average over. Ex p = 0.3 means average over last 30% of observations.
    Methods:
        Suffix averaging: Average over last p percentage of observations.
        Uniform: Average over all iterates equally
        Last iterate: Consider only the last iterate.
        """
    n = len(array)
    # Check p is a proportion
    if p is None:
        p = 0.5
    if not (0 <= p <= 1):
        raise ValueError("p must be between 0 and 1")

    if method == "suffix_averaging":
        i = round((1-p)*n)
        result = np.mean(array[i:], axis=0)
    elif method == "uniform" or method is None:
        result = np.mean(array, axis=0)
    elif method == "last_iterate":
        last_iterate = n - 1
        result = array[last_iterate]
    else:
        raise ValueError("Method must be one of the options")
    return result

In [31]:
def log_loss(betas,data_x,data_y):
    return sum(np.logaddexp(0,-data_y*np.inner(data_x,betas)))

In [32]:
def generate_y(n,x, betas):
    y = np.zeros(n)
    for t in range(n):
        prob_y1 = sig(np.inner(x[t],betas))
        y[t] = np.random.choice([1,-1], size = 1, p = [prob_y1, 1-prob_y1]).item()
    return y

## Data to test algorithm functions

In [33]:
# Generate y randomly
n = 100
d = 3
mean = [rng.normal(20,24,1).item(), rng.normal(14,40,1).item(), rng.normal(0,5,1).item()]
cov = [[1, 0, 0], [0, 3, 0], [0,0,2]]  # diagonal covariance, arbitrary
d = np.shape(cov)[1]

prob_y1 = rng.uniform(0.1,0.9,1).item()

y = np.random.choice([1,-1], size = n, p = [prob_y1, 1-prob_y1])

x = np.random.multivariate_normal(mean, cov, size = n)


In [34]:
#Generate y depending on x
# Generate y randomly
n = 100
d = 3

test_size = round(n*0.2)
train_size = n - test_size

mean = [rng.normal(0.2,2,1).item(), rng.normal(1,.4,1).item(), rng.normal(0,5,1).item()]
cov = [[1, 0, 0], [0, 3, 0], [0,0,2]]  # diagonal covariance, arbitrary
d = np.shape(cov)[1]

x = np.random.multivariate_normal(mean, cov, size = n)

betas = [-0.1,1.4,0.2] # random numbers

y = generate_y(n,x,betas)

#Separate test and train data
x_train = x[:train_size]
y_train = y[:train_size]

x_test = x[train_size:]
y_test = y[train_size:]


## Online Gradient Descent

In [35]:
def online_gd(n, d, data_y, data_x, B, OTB_method = None, OTB_average_proportion = None):
    """Online gradient descent
    Parameters
    ---------
    n : float
        Size of the dataset
    d : float
        Dimension of the context vector, i.e. number of independent variables
    data_y : ndarray
        Vector with outcome(y) values. Dimensions: n x 1
    data_x: ndarray
        Array with independent variables. Dimensions: n x d
    eta: float
        Learning rate.
    B: float
        Radius of the L2 ball constraint for the parameter vector beta
    OTB_method: string
        Averaging method, default none which implies the averaging function default which is uniform.
    OTB_average_proportion:
    """

    # Start time
    start_time = datetime.now()

    # Initialize vectors
    betas = np.zeros((n,d)) # Parameter vector of all n rounds.

    loss = np.zeros(n)
    G = 1 # Max gradient
    eta = 1
    gradient = np.zeros(d)

    for t in range(n):
        # Observe context vector for this round
        xt = data_x[t]

        # Update parameter (beta)
        if t == 0:
            beta_tilde = np.zeros(d)
        else:
            beta_tilde = betas[t-1] - eta*gradient

        # Euclidean projection of beta onto the L2 ball
        #betas[t] = project_onto_l2_ball(beta_tilde,B)
        betas[t] = beta_tilde

        # Issue prediction
        #y_hat = sig(np.inner(betas[t],xt)) commented out because its never used so there's no need to calculate it

        # Observe true y from the data
        yt = data_y[t]

        # Calculate gradient and sum of gradients
        gradient = -yt*xt*sig(-yt*np.inner(xt,betas[t]))
        # update maximum gradient
        G = np.max([G,np.linalg.norm(gradient)**2])

        # Suffer loss
        loss[t] = np.logaddexp(0,-yt*np.inner(xt,betas[t]))

        # Recalculate eta
        eta = 1 / np.sqrt((t+1) * G) # Here t+1 simply because python initializes t to 0


    # Store endtime
    end_time = datetime.now()

    # Get Batch estimate
    betas_OTB = calculate_OTB(betas, OTB_method, OTB_average_proportion)

    # Get Batch estimate loss
    loss_OTB =  log_loss(betas_OTB,data_x, data_y)

    return{
        "runtime" : (end_time - start_time).total_seconds(),
        "Total_Loss" : loss_OTB,
        "Online_Loss" : sum(loss),
        "Batch_estimate": betas_OTB,
        "All_Betas": betas
    }


## Testing ODG

In [36]:
ogd = online_gd(n = n,d = d,data_y = y,data_x = x,B = 40)

## AIOLI

In [37]:
def AIOLI(n, d, data_y, data_x, B, X, OTB_method = None, OTB_average_proportion = None):
    """AIOLI
    This is the AIOLI using the reduced version of the loss function.

    Parameters
    ---------
    n : float
        Size of the dataset
    d : float
        Dimension of the context vector, i.e. number of independent variables
    data_y : ndarray
        Vector with outcome(y) values. Dimensions: n x 1
    data_x: ndarray
        Array with independent variables. Dimensions: n x d
    B: float
        Radius of the L2 ball constraint for the parameter vector beta
    X: float
        Radius of the L2 ball constraint for the feature vector x"""
    # Record start time
    start_time = datetime.now()

    # Initialize vectors for storage
    betas = np.zeros((n, d))
    ys = np.zeros(n)
    y_hats = np.zeros(n)
    loss = np.zeros(n)
    etas = np.zeros(n)
    gradients = np.zeros((n, d))
    reg_lambda = 1 / (B**2)

    # Keep sums for loss functions
    gradients_sum = 0
    first_sum  = 0
    etas_gradients = 0
    convergence = 0

    # Define function to minimize for AIOLI
    def parameter_function_AIOLI(b, xt, reg_lambda, first_sum, etas_gradients):
        l_hat = np.inner(b,first_sum) + np.inner(np.inner(b,etas_gradients), b)
        # Use function np.logaddexp = log(exp(x1) + exp(x2)) to ensure numerical stability, not sure how it works
        result = l_hat + np.logaddexp(0,np.inner(b,xt)) + np.logaddexp(0,np.inner(-b,xt)) + reg_lambda*np.linalg.norm(b)**2
        return result

    # Run Algorithm
    for t in range(n):
        # Get context vector
        xt = data_x[t]

        # Update beta parameter
        if t == 0: # Initialize to 0 for first round
            beta_tilde = np.zeros(d)
        else:
            x0 = np.zeros(d) # Initializer for minimizing algorithm
            result = minimize(parameter_function_AIOLI, x0, method = "l-bfgs-b", args=(xt, reg_lambda,first_sum, etas_gradients))
            beta_tilde = result.x
            convergence += result.success

        # Project beta onto set
        #betas[t] = project_onto_l2_ball(beta_tilde,B)
        betas[t] = beta_tilde

        # Generate prediction
        y_hats[t] = np.inner(xt,betas[t])

        # Observe true y from data
        ys[t] = data_y[t]

         # Compute gradient
        gradients[t] = -ys[t]*xt*sig(-ys[t]*np.inner(xt,betas[t]))

        # Estimate eta
        etas[t] = np.exp(ys[t]*y_hats[t])/(1+B*X)

        # Suffer loss
        loss[t] = np.logaddexp(0,-ys[t]*np.inner(xt,betas[t]))

        # Compute elements for AIOLI loss function
        gradients_sum += gradients[t]
        first_sum += gradients[t]*(1 - etas[t]*np.inner(betas[t],gradients[t]))
        etas_gradients += (etas[t]/2)*np.inner(gradients[t],gradients[t])

    # for loop ends

    # Get Batch estimate
    betas_OTB = calculate_OTB(betas, OTB_method, OTB_average_proportion)

    # Get Batch estimate loss
    loss_OTB = log_loss(betas_OTB,data_x, data_y)

    end_time = datetime.now()
    return{
        "runtime" : (end_time - start_time).total_seconds(),
        "Total_Loss" : loss_OTB,
        "Online_Loss" : sum(loss),
        "Batch_estimate": betas_OTB,
        "All_Betas": betas,
        "convergence" : convergence
    }



## Testing AIOLI

In [38]:
aioli = AIOLI(n = n,d = d,data_y = y,data_x = x,X = 50,B = 40)

## Logistic Regression

In [39]:
def classic_logistic_regression(data_x,data_y): #Using scikit learn
    start_time = datetime.now()
    # Initialize the model
    logreg = LogisticRegression(penalty=None, fit_intercept=False)

    # fit the model with data
    logreg.fit(data_x,data_y)
    betas = logreg.coef_.flatten() # flatten is To ensure it's a one dimensional array of d elements

    loss = log_loss(betas, data_x, data_y)
    end_time = datetime.now()
    return{
        "runtime" : (end_time - start_time).total_seconds(),
        "Total_Loss" : loss,
        "Batch_estimate": betas
    }

## Ridge Logistic Regression

In [40]:
def ridge_logistic_regression(data_x,data_y): #Using scikit learn
    start_time = datetime.now()
    # Initialize the model
    logreg = LogisticRegression(penalty="l2", random_state=1, fit_intercept=False)

    # fit the model with data
    logreg.fit(data_x,data_y)
    betas = logreg.coef_.flatten() # flatten is To ensure it's a one dimensional array of d elements

    loss = log_loss(betas, data_x, data_y)
    end_time = datetime.now()
    return{
        "runtime" : (end_time - start_time).total_seconds(),
        "Total_Loss" : loss,
        "Batch_estimate": betas
    }

# Testing algorithm functions

## Test performance of algorithms: runtime and loss

In [41]:
runs = 20
n = 1000 # Sample size, equivalent to T in online learning
results_all = []
test_set_results = []

# All parameters for data generation and for the upper bounds of beta and context vector are just arbitrary numbers

for i in range(runs):
    test_size = round(n*0.2)
    train_size = n - test_size

    mean = [rng.normal(0.2,2,1).item(), rng.normal(1,.4,1).item(), rng.normal(0,5,1).item()]
    cov = [[1, 0, 0], [0, 3, 0], [0,0,2]]  # diagonal covariance, arbitrary
    d = np.shape(cov)[1]

    x = np.random.multivariate_normal(mean, cov, size = n)

    betas = [-0.1,1.4,0.2] # random numbers

    y = generate_y(n,x,betas)

    #Separate test and train data
    x_train = x[:train_size]
    y_train = y[:train_size]

    x_test = x[train_size:]
    y_test = y[train_size:]

    # Run algorithms
    ogd = online_gd(n = train_size,d = d,data_y = y_train,data_x = x_train,B = 40)
    ai  = AIOLI(n = train_size,d = d,data_y = y_train,data_x = x_train, B = 10, X = 15)
    classic_lg = classic_logistic_regression(data_x = x_train, data_y = y_train)
    ridge_lg = ridge_logistic_regression(data_x = x_train, data_y = y_train)

    # Get different batch estimates for the Online Algorithms
    OGD_OTB_uniform = calculate_OTB(ogd["All_Betas"], "uniform")
    OGD_OTB_Suffix = calculate_OTB(ogd["All_Betas"], "suffix_averaging")
    OGD_OTB_lastit = calculate_OTB(ogd["All_Betas"], "last_iterate")

    AIOLI_OTB_uniform = calculate_OTB(ai["All_Betas"], "uniform")
    AIOLI_OTB_Suffix = calculate_OTB(ai["All_Betas"], "suffix_averaging")
    AIOLI_OTB_lastit = calculate_OTB(ai["All_Betas"], "last_iterate")

    # Get Training errors
    # Get losses
    loss_OGD_Suffix = log_loss(OGD_OTB_Suffix, x_train, y_train)
    loss_OGD_lastit = log_loss(OGD_OTB_lastit, x_train, y_train)

    loss_AIOLI_Suffix = log_loss(AIOLI_OTB_Suffix, x_train, y_train)
    loss_AIOLI_lastit = log_loss(AIOLI_OTB_lastit, x_train, y_train)

    # Add results to dataframe

    results_all.append({
        "algorithm": "OGD_uniform",
        "runtime": ogd["runtime"],
        "loss (trainset)": ogd["Total_Loss"],
        "online_loss": ogd["Online_Loss"],
        "betas": ogd["Batch_estimate"]
    })

    results_all.append({
        "algorithm": "OGD_Suffix",
        "loss (trainset)": loss_OGD_Suffix,
        "betas": OGD_OTB_Suffix,
    })

    results_all.append({
        "algorithm": "OGD_Last_iterate",
        "loss (trainset)": loss_OGD_lastit,
        "betas": OGD_OTB_lastit,
    })

    results_all.append({
        "algorithm": "AIOLI_uniform",
        "runtime": ai["runtime"],
        "loss (trainset)": ai["Total_Loss"],
        "online_loss": ai["Online_Loss"],
        "betas": ai["Batch_estimate"],
        "convergence": ai["convergence"]
    })

    results_all.append({
        "algorithm": "AIOLI_Suffix",
        "loss (trainset)": loss_AIOLI_Suffix,
        "betas": AIOLI_OTB_Suffix,
    })

    results_all.append({
        "algorithm": "AIOLI_lastit",
        "loss (trainset)": loss_AIOLI_lastit,
        "betas": AIOLI_OTB_lastit,
    })

    results_all.append({
        "algorithm": "Classical_LG",
        "runtime": classic_lg["runtime"],
        "loss (trainset)": classic_lg["Total_Loss"],
        "betas": classic_lg["Batch_estimate"]
    })

    results_all.append({
        "algorithm": "Ridge_LG",
        "runtime": ridge_lg["runtime"],
        "loss (trainset)": ridge_lg["Total_Loss"],
        "betas": ridge_lg["Batch_estimate"]
    })


    # Get test errors
    # Get losses
    loss_OGD_uniform_test =log_loss(OGD_OTB_uniform, x_test, y_test)
    loss_OGD_Suffix_test = log_loss(OGD_OTB_Suffix, x_test, y_test)
    loss_OGD_lastit_test = log_loss(OGD_OTB_lastit, x_test, y_test)

    loss_AIOLI_uniform_test = log_loss(AIOLI_OTB_uniform, x_test, y_test)
    loss_AIOLI_Suffix_test = log_loss(AIOLI_OTB_Suffix, x_test, y_test)
    loss_AIOLI_lastit_test = log_loss(AIOLI_OTB_lastit, x_test, y_test)

    loss_LG_test = log_loss(classic_lg["Batch_estimate"], x_test, y_test)
    loss_LG_Ridge_test = log_loss(ridge_lg["Batch_estimate"], x_test, y_test)

    # Add results to dataframe

    test_set_results.append({
        "algorithm": "OGD_uniform",
        "loss (testset)": loss_OGD_uniform_test
    })

    test_set_results.append({
        "algorithm": "OGD_Suffix",
        "loss (testset)": loss_OGD_Suffix_test
    })

    test_set_results.append({
        "algorithm": "OGD_Last_iterate",
        "loss (testset)": loss_OGD_lastit_test
    })

    test_set_results.append({
        "algorithm": "AIOLI_uniform",
        "loss (testset)": loss_AIOLI_uniform_test
    })

    test_set_results.append({
        "algorithm": "AIOLI_Suffix",
        "loss (testset)": loss_AIOLI_Suffix_test
    })

    test_set_results.append({
        "algorithm": "AIOLI_lastit",
        "loss (testset)": loss_AIOLI_lastit_test
    })

    test_set_results.append({
        "algorithm": "Classical_LG",
        "loss (testset)": loss_LG_test
    })

    test_set_results.append({
        "algorithm": "Ridge_LG",
        "loss (testset)": loss_LG_Ridge_test
    })


final_results = pd.DataFrame(results_all)
test_results = pd.DataFrame(test_set_results)


In [42]:
#final_results

In [43]:
summary = final_results.groupby("algorithm")[["runtime", "loss (trainset)", "online_loss", "betas"]].mean()
summary["betas"] = summary["betas"].apply(lambda array: np.round(array, 3)) # To round betas
summary

,runtime,loss (trainset),online_loss,betas
algorithm,,,,
AIOLI_Suffix,NaN,312.504004,NaN,"[-0.128, 2.299, 0.277]"
AIOLI_lastit,NaN,328.550287,NaN,"[-0.206, 1.919, 0.354]"
AIOLI_uniform,0.985640,416.525447,373.479902,"[-0.126, 3.753, 0.521]"
Classical_LG,0.002061,281.371473,NaN,"[-0.099, 1.395, 0.189]"
OGD_Last_iterate,NaN,286.629023,NaN,"[-0.068, 1.218, 0.156]"
OGD_Suffix,NaN,285.654859,NaN,"[-0.074, 1.197, 0.167]"
OGD_uniform,0.007373,290.141317,301.496429,"[-0.073, 1.124, 0.168]"
Ridge_LG,0.001805,281.383314,NaN,"[-0.097, 1.379, 0.188]"


In [44]:
summary_test = test_results.groupby("algorithm")[["loss (testset)"]].mean()
summary_test

,loss (testset)
algorithm,
AIOLI_Suffix,76.353175
AIOLI_lastit,76.864848
AIOLI_uniform,102.893530
Classical_LG,68.708580
OGD_Last_iterate,69.253856
OGD_Suffix,69.626226
OGD_uniform,71.066449
Ridge_LG,68.706792


### Check AIOLI convergence

In [45]:
round((final_results[final_results["algorithm"] == "AIOLI_uniform"]["convergence"].mean() * 100 / n).item(),2)

79.9

# Ignore from here on, it's simply code to test some functions/debugging

## Tests

In [46]:
#etas = np.array([1/np.sqrt(3),1/np.sqrt(3),1/np.sqrt(3)])
#xt = np.array([1,2,3])
#betas = np.array([[2,2,5],[1,1,3],[4,5,6]])
#reg_lambda = 0.3
#gradients = np.array([[2,2,5],[1,1,3],[4,5,6]])
#b = np.array([1,2,3])
#sum_loss = 5


#(sum(etas)/2)*np.inner((b-betas[2]),gradients[2])*np.inner(gradients[2],(b-betas[2]))
#np.linalg.norm(etas)


In [47]:
#diff = b - betas
#inner_prod = np.sum(gradients * diff,axis = 1)
#np.sum(np.sum(etas)/2 * inner_prod**2)

In [48]:
x0 = np.array([0,0,0])

#res = minimize(parameter_function_AIOLI, x0, method = "L-BFGS-B", args=(xt, reg_lambda, sum_loss,first_sum, etas_gradients))
#parameter_function_AIOLI(x0, xt, betas, etas, reg_lambda, sum_loss,gradients)
#res.x

In [49]:
diff = b - betas
gs_bs = np.sum(gradients * diff,axis = 1)
quadratic = np.sum(np.sum(etas)/2 * gs_bs**2)

l_hat = sum_loss + np.sum(gs_bs) + quadratic

l_hat + np.log(1+np.exp(np.inner(b,xt))) + np.log(1+np.exp(np.inner(-b,xt))) + reg_lambda*np.linalg.norm(b)

NameError: name 'b' is not defined

## Generate data

In [ ]:
n = 1000

mean = [20, 5, 2]
cov = [[1, 0, 0], [0, 3, 0], [0,0,2]]  # diagonal covariance
d = np.shape(cov)[1]
y = np.random.binomial(n=1, p=0.3, size=n)

x1, x2, x3 = np.random.multivariate_normal(mean, cov, size = n).T
plt.plot(x1, x2, 'x')
plt.axis('equal')
plt.show()

plt.plot(x1, x3, 'x')
plt.axis('equal')
plt.show()


In [ ]:
x = np.random.multivariate_normal(mean, cov, size = n)
sns.countplot(x=y)
plt.show()


In [ ]:
np.log(1000)

## OTB conversion proof tests

In [ ]:
weights = np.array([0.3,0.7])
values = np.array([2,3])

print(np.sum(values))
print(np.sum(weights*values))